# 📓 실습 02. MLflow 로컬 Tracking API 연동 및 실험 로깅

이 실습에서는 로컬 환경에서 MLflow 실험 관리 데몬을 구동하고, 가상의 반도체 CMP 고장 예지보전 XGBoost 모델의 하이퍼파라미터 및 손실 평가지표(RMSE, R2)를 중앙 레지스트리에 자동 기록하고 버전을 할당하는 작업을 수행합니다.

In [ ]:
import os
import json
import time

# 1. MLflow 로컬 모의 레코더 클래스 구현
# (mlflow가 설치되지 않은 환경에서도 동작성 보장)
class MockMLflowTracker:
    def __init__(self):
        self.run_id = "run_cmp_" + str(int(time.time()))
        self.params = {}
        self.metrics = {}
        self.registered_model_version = "v1.0.0"
        
    def log_param(self, key, value):
        self.params[key] = value
        
    def log_metric(self, key, value):
        self.metrics[key] = value
        
    def save_run(self):
        output = {
            "run_id": self.run_id,
            "parameters": self.params,
            "metrics": self.metrics,
            "model_version": self.registered_model_version,
            "status": "FINISHED"
        }
        os.makedirs("mlruns", exist_ok=True)
        with open(os.path.join("mlruns", "run_summary.json"), "w") as f:
            json.dump(output, f, indent=2)
        return output

tracker = MockMLflowTracker()
print(f"MLflow 실험 가동 준비 완료 (Run ID: {tracker.run_id})")

In [ ]:
# 2. 하이퍼파라미터 및 평가지표 로깅 수행
print("[MLflow] XGBoost Classifier 훈련 로그 수집 중...")
tracker.log_param("max_depth", 6)
tracker.log_param("learning_rate", 0.05)
tracker.log_param("n_estimators", 150)
tracker.log_param("imbalance_ratio", "100:1")

tracker.log_metric("train_loss", 0.0825)
tracker.log_metric("val_loss", 0.1140)
tracker.log_metric("f1_score", 0.9421)
tracker.log_metric("recall", 0.9250)

summary = tracker.save_run()
print("\n🎉 MLflow 저장 완료! 실험 데이터 요약:")
print(json.dumps(summary, indent=2))